# Model Config

Retrieve the provider's reported defaults for the configured LM
(context window, loaded state, family) and emit a
`recommended_max_tokens` grounded in that real context window.

Per-model skill notebooks should start by calling this skill so they
don't hard-code values that will silently truncate. The result is
written to `outputs/result.json` as durable provenance for DSPy GEPA.

Backed by `notebook_agent.model_info.model_info()`, which probes
LM Studio's `/api/v0/models/{id}` first and falls back to the
standard OpenAI `/v1/models` listing.


In [ ]:
# parameters
task_id = None
parent_task_id = None
input_payload = {}
output_dir = "./outputs"
run_dir = "."
budget = {}
model = None
base_url = None
headroom_tokens = 1024

In [ ]:
import json
from pathlib import Path
Path(output_dir).mkdir(parents=True, exist_ok=True)


In [ ]:
# Look up the provider's reported defaults for this model.
from notebook_agent.litellm_client import LiteLLMClient
from notebook_agent.model_info import model_info
from notebook_agent.notebook_init import get_notebook_config

_cfg_client = None
try:
    _cfg_client = get_notebook_config().client
except Exception:
    _cfg_client = None

_client = LiteLLMClient(
    model=model or (_cfg_client.model if _cfg_client else None),
    base_url=base_url or (_cfg_client.base_url if _cfg_client else None),
    api_key=(_cfg_client.api_key if _cfg_client else None),
    provider=(_cfg_client.provider if _cfg_client else None),
)

_info = model_info(_client)
if _info is None:
    result = {
        'id': _client.model,
        'base_url': _client.base_url,
        'error': 'model_info_unavailable',
    }
else:
    result = _info.to_dict()
    result['recommended_max_tokens'] = _info.recommended_max_tokens(headroom=headroom_tokens)
result


In [ ]:
import json, os
from pathlib import Path
_output_dir = Path(output_dir)
_output_dir.mkdir(parents=True, exist_ok=True)
_payload = result
(_output_dir / "result.json").write_text(json.dumps(_payload, indent=2))


In [ ]:
import json
from pathlib import Path
_run_dir = Path(run_dir)
_mp = _run_dir / 'manifest.json'
if _mp.exists():
    _m = json.loads(_mp.read_text())
    _m.setdefault('outputs', {})['result_json'] = str(Path(output_dir, 'result.json'))
    _m.setdefault('notebook', {})['skill_id'] = 'core.model_config'
    _mp.write_text(json.dumps(_m, indent=2))


In [ ]:
assert isinstance(result, dict), 'result must be a dict'
assert (Path(output_dir) / 'result.json').exists()
